# Bidirectional Conversion Between Inspired CO₂ Fraction and Arterial CO₂ Partial Pressure
### A ventilation-aware technical note with reference implementation

---

**Abstract.** Carbon-dioxide challenges are reported either as an inspired fraction
($F_iCO_2$, per cent) or as a partial pressure ($P_aCO_2 \approx P_{ET}CO_2$, mmHg or kPa).
The two are not interconvertible by simple proportionality: raising inspired CO₂ elevates
arterial CO₂, but the resulting hypercapnia augments ventilation and eliminates part of the
added load. This note reconstructs the alveolar CO₂ mass balance with an explicit inspired
term, couples it to a linear hypercapnic ventilatory response, and reduces the result to a
single governing equation that is solved in closed form in both directions. Each step is
paired with the corresponding calculation so the derivation and its numerical behaviour can
be inspected together. All constants are referenced; the sources are collected in the
accompanying library (*CO2 Ventilation* collection).

## 1. Physiological basis

The dependence of alveolar gas composition on ventilation is among the oldest quantitative
results in respiratory physiology, and the working equation used here follows from three
classical results.

**(a) Ventilation determines alveolar CO₂.** Haldane and Priestley (1905) established that
alveolar CO₂ is closely regulated and that ventilation is governed chiefly by CO₂. In the
steady state, metabolic CO₂ production ($\dot V_{CO_2}$) is balanced by pulmonary elimination,
so the alveolar CO₂ fraction equals output divided by alveolar ventilation:

$$F_ACO_2 = \frac{\dot V_{CO_2}}{\dot V_A}. \qquad (1)$$

**(b) The ideal alveolar compartment.** Riley and Cournand (1949) formalised a single,
well-mixed *ideal alveolar air* in which, for healthy lungs, arterial and end-tidal tensions
equal the alveolar value, $P_aCO_2 \approx P_ACO_2 \approx P_{ET}CO_2$ — the same premise that
underlies contemporary end-tidal targeting (Slessarev et al., 2007).

**(c) From fraction to partial pressure.** By Dalton's law of partial pressures (Dalton,
1802), each species exerts a pressure equal to its fraction multiplied by the total dry-gas
pressure; in humidified alveolar gas at 37 °C the water-vapour pressure ($P_{H_2O} = 47$ mmHg)
is first subtracted, giving $P_ACO_2 = F_ACO_2 \cdot (P_{atm} - P_{H_2O})$. Applying this to
Eq. (1) yields the classical alveolar ventilation equation (West, 2012; Lumb & Thomas, 2020):

$$P_ACO_2 = \frac{K \cdot \dot V_{CO_2}}{\dot V_A}, \qquad K = 0.863. \qquad (2)$$

The factor $K$ is a unit constant only: $\dot V_{CO_2}$ is conventionally reported at STPD and
$\dot V_A$ at BTPS, and $K = \dfrac{310}{273} \cdot \dfrac{760}{1000} = 0.863$ carries the
temperature and pressure corrections (Cruickshank & Hirschauer, 2004; West, 2012).

Equation (2) was formulated for carbon-dioxide-free inspirate and for a *fixed* alveolar
ventilation. A CO₂ challenge violates both premises. Sections 2.1–2.4 reinstate the inspired
term and the ventilatory response, and solve the resulting relationship in closed form.

In [ ]:
# Eq. (2): the classical alveolar ventilation equation, CO2-free inspirate.
K, VCO2 = 0.863, 200.0        # unit constant; CO2 output, mL/min STPD
VA = 4.3                       # resting alveolar ventilation, L/min BTPS
PACO2 = K * VCO2 / VA          # mmHg
print(f'PACO2 = K*VCO2/VA = {K}*{VCO2}/{VA} = {PACO2:.1f} mmHg   (normocapnic ~40)')

## 2. The conversion, step by step

Each step below states the reasoning and equation, then evaluates it immediately.

### 2.1 Step 1 — Baseline alveolar ventilation from the baseline PaCO₂

The model is anchored to the subject's normocapnic state. Rearranging Eq. (2) at baseline
gives the baseline alveolar ventilation directly from the baseline arterial tension:

$$\dot V_{A,\text{base}} = \frac{K \cdot \dot V_{CO_2}}{P_aCO_{2,\text{base}}}. \qquad (3)$$

For $\dot V_{CO_2} = 200$ mL·min⁻¹ and $P_aCO_{2,\text{base}} = 40$ mmHg this is ≈ 4.3 L·min⁻¹,
the accepted resting value (West, 2012; Lumb & Thomas, 2020). Anchoring in this way renders
the mapping self-consistent: at $F_iCO_2 = 0$ it returns the baseline exactly. Where the
baseline is not reported, $P_aCO_{2,\text{base}} = 40$ mmHg is assumed together with the
literature resting value $\dot V_{A,\text{base}} = 4.2$ L·min⁻¹ (West, 2012; Lumb & Thomas,
2020); the ventilatory feedback of Step 3 absorbs the small resulting inconsistency.

In [ ]:
# Eq. (3): baseline alveolar ventilation from the baseline PaCO2.
PaCO2_base = 40.0                              # mmHg, normocapnic baseline
VA_base = K * VCO2 / PaCO2_base                # L/min BTPS
print(f'VA_base = K*VCO2/PaCO2_base = {K}*{VCO2}/{PaCO2_base:.0f} = {VA_base:.3f} L/min')

### 2.2 Step 2 — Inclusion of inspired CO₂ in the mass balance

When the inspirate contains CO₂, the alveolar balance acquires an inspired term
(Slessarev et al., 2007):

$$F_ACO_2 = F_iCO_2 + \frac{\dot V_{CO_2}}{\dot V_A}. \qquad (4)$$

Multiplying throughout by $(P_{atm} - P_{H_2O})$ and applying Dalton's law converts every term
to a partial pressure (Lumb & Thomas, 2020):

$$P_ACO_2 = \underbrace{F_iCO_2 \cdot (P_{atm} - P_{H_2O})}_{P_iCO_2} + \frac{K \cdot \dot V_{CO_2}}{\dot V_A}. \qquad (5)$$

Evaluating Eq. (5) with ventilation held at its baseline value gives the *feedback-free*
estimate, which markedly overestimates the true arterial tension and thereby motivates
Step 3.

In [ ]:
# Eq. (5): alveolar CO2 with an inspired term, ventilation held at baseline.
Patm, PH2O = 760.0, 47.0
Pdry = Patm - PH2O                             # dry-gas pressure, 713 mmHg
FiCO2 = 0.05                                   # inspired fraction, 5%
PICO2 = FiCO2 * Pdry                           # inspired partial pressure, mmHg
PACO2_nofeedback = PICO2 + K * VCO2 / VA_base  # VA fixed at baseline
print(f'PICO2               = FiCO2*(Patm-PH2O) = {PICO2:.1f} mmHg')
print(f'PACO2 (no feedback) = {PACO2_nofeedback:.1f} mmHg   <- overestimate; see Step 3')

### 2.3 Step 3 — The hypercapnic ventilatory response

Alveolar ventilation is not fixed: elevated $P_aCO_2$ increases it. Over the near-linear
working range (≈ 40–80 mmHg) this hypercapnic ventilatory response is represented by a
straight line of slope $S$ (Read, 1967; Hirshman et al., 1975; West, 2012):

$$\dot V_A(P_aCO_2) = \dot V_{A,\text{base}} + S \cdot (P_aCO_2 - P_aCO_{2,\text{base}}). \qquad (6)$$

The population-mean slope $S = 2.69$ L·min⁻¹·mmHg⁻¹ is adopted (Hirshman et al., 1975; 44
healthy adults, range 1.16–5.95), consistent with the rebreathing slopes of 1.96, 2.99 and
4.04 reported by Read (1967). Its variability is the dominant source of uncertainty and is
examined in Section 5.

In [ ]:
# Eq. (6): CO2-sensitive alveolar ventilation.
S = 2.69                                       # HCVR slope, L/min/mmHg
def VA_of(paco2):
    return VA_base + S * (paco2 - PaCO2_base)
for paco2 in (40.0, 45.0, 60.0):
    print(f'PaCO2 {paco2:5.1f} mmHg  ->  VA = {VA_of(paco2):6.2f} L/min')

### 2.4 Step 4 — Governing equation and its two directions

Substituting the CO₂-sensitive ventilation, Eq. (6), into the balance, Eq. (5), with
$P_aCO_2 \approx P_ACO_2$, gives the single governing equation:

$$P_aCO_2 = F_iCO_2 \cdot (P_{atm} - P_{H_2O}) + \frac{K \cdot \dot V_{CO_2}}{\dot V_{A,\text{base}} + S \cdot (P_aCO_2 - P_aCO_{2,\text{base}})}. \qquad (7)$$

**Forward direction ($P_aCO_2 \rightarrow F_iCO_2$)** is explicit:

$$F_iCO_2 = \frac{P_aCO_2 - K \cdot \dot V_{CO_2} / \dot V_A(P_aCO_2)}{P_{atm} - P_{H_2O}}. \qquad (8)$$

**Reverse direction ($F_iCO_2 \rightarrow P_aCO_2$)** carries $P_aCO_2$ on both sides; because
$\dot V_A$ is linear in $P_aCO_2$, Eq. (7) is a quadratic with an exact physical (upper) root,
so no iteration is required:

$$P_aCO_2 = \frac{(S \cdot P_iCO_2 - D) + \sqrt{(S \cdot P_iCO_2 - D)^2 + 4 \cdot S \cdot (P_iCO_2 \cdot D + K \cdot \dot V_{CO_2})}}{2 \cdot S},\qquad D \equiv \dot V_{A,\text{base}} - S \cdot P_aCO_{2,\text{base}}. \qquad (9)$$

The reference implementation evaluates both roots below. With the feedback restored, the 5%
inspired challenge that Step 2 mapped to ≈ 76 mmHg resolves to ≈ 45 mmHg, and the forward
solution recovers the original fraction (round-trip check).

In [ ]:
from fico2_paco2_converter import (
    params_from_baseline, fico2_to_paco2, paco2_to_fico2, mmhg_to_kpa,
)

p = params_from_baseline(40.0)                 # Step 1 anchoring, VA_base self-consistent
pa = fico2_to_paco2(5.0, p)                    # reverse: Eq. (9)
fi = paco2_to_fico2(pa, p)                     # forward: Eq. (8), round-trip
print(f'FiCO2 5.00 %  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.2f} kPa')
print(f'PaCO2 {pa:.2f} mmHg  ->  FiCO2 = {fi:.2f} %   (recovers input)')

## 3. Using the converter

The two cells below constitute the reusable tool. Set the inputs and run each cell.

**Direction 1 — a target $P_aCO_2$ is known; estimate $F_iCO_2$.** Provide the target
(hypercapnic) tension and the baseline; Step 1 fixes $\dot V_{A,\text{base}}$ and Eq. (8)
returns the required inspired fraction.

In [ ]:
paco2_target = 50.0   # mmHg, hypercapnic level to reach
paco2_base   = 40.0   # mmHg, normocapnic baseline

p = params_from_baseline(paco2_base)
fico2 = paco2_to_fico2(paco2_target, p)
print(f'VA_base = {p.VA_base:.3f} L/min ;  VA(target) = {p.VA(paco2_target):.3f} L/min')
print(f'PaCO2 {paco2_target:.1f} mmHg  ->  FiCO2 = {fico2:.3f} %'      f'   (PICO2 = {fico2/100*p.Pdry:.1f} mmHg)')

**Direction 2 — an $F_iCO_2$ is known; estimate $P_aCO_2$.** If the baseline is known, set
it; otherwise leave `paco2_base = None` for the literature resting fallback
($P_aCO_{2,\text{base}} = 40$ mmHg, $\dot V_{A,\text{base}} = 4.2$ L·min⁻¹). The reverse
solution uses the closed form, Eq. (9).

In [ ]:
from fico2_paco2_converter import params_resting_default

fico2_percent = 5.0    # inspired CO2, %
paco2_base    = None   # mmHg, or None if unknown -> resting fallback

if paco2_base is None:
    p = params_resting_default()
    tag = f'resting fallback (base={p.PaCO2_base:.0f} mmHg, VA_base={p.VA_base:.1f} L/min)'
else:
    p = params_from_baseline(paco2_base)
    tag = f'baseline known (base={p.PaCO2_base:.0f} mmHg, VA_base={p.VA_base:.3f} L/min)'

pa = fico2_to_paco2(fico2_percent, p)
print(f'mode: {tag}')
print(f'FiCO2 {fico2_percent:.2f} %  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.3f} kPa'      f'   ({pa - p.PaCO2_base:+.2f} mmHg from baseline)')

## 4. Reference values

Steady-state $P_aCO_2$ for a fixed inspired CO₂ at sea level (baseline 40 mmHg, $S = 2.69$).

In [ ]:
p = params_from_baseline(40.0)
print(f"{'FiCO2 %':>8} {'PICO2 mmHg':>11} {'PaCO2 mmHg':>11} {'PaCO2 kPa':>10}")
for fi in (0, 2, 5, 7, 8):
    pa = fico2_to_paco2(fi, p)
    print(f'{fi:8.0f} {fi/100*p.Pdry:11.1f} {pa:11.2f} {mmhg_to_kpa(pa):10.2f}')

## 5. Sensitivity to the ventilatory-response slope

The slope $S$ is the principal source of uncertainty. Hirshman et al. (1975) report a mean
of 2.69 with a wide individual range (1.16–5.95); the two-component central and peripheral
structure of the response is characterised by Mohan and Duffin (1997) and Pedersen et al.
(1999). Mapping a fixed $F_iCO_2$ across this range quantifies the dependence of the estimate
on the assumed slope.

In [ ]:
fico2 = 5.0
print(f'FiCO2 = {fico2}% , baseline 40 mmHg\n')
print(f"{'S':>6} {'PaCO2 mmHg':>11}")
for S in (1.16, 1.96, 2.69, 4.04, 5.95):
    p = params_from_baseline(40.0, S=S)
    print(f'{S:6.2f} {fico2_to_paco2(fico2, p):11.2f}')

In [ ]:
# Optional figure (requires matplotlib): FiCO2 -> PaCO2 across a range of S.
try:
    import numpy as np, matplotlib.pyplot as plt
    grid = np.linspace(0, 8, 81)
    plt.figure(figsize=(6, 4))
    for S in (1.16, 2.69, 5.95):
        p = params_from_baseline(40.0, S=S)
        plt.plot(grid, [fico2_to_paco2(f, p) for f in grid], label=f'S = {S}')
    plt.xlabel('FiCO$_2$ (%)'); plt.ylabel('PaCO$_2$ (mmHg)')
    plt.title('FiCO$_2$ $\\rightarrow$ PaCO$_2$ across the HCVR slope range')
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed - skipping figure (pip install matplotlib)')

## 6. Interactive prompt

The following cell reproduces the command-line interface
(`python fico2_paco2_converter.py -i`), prompting for the direction and the required inputs.

In [ ]:
from fico2_paco2_converter import interactive
interactive()

## 7. Assumptions and limitations

- Ideal alveolar–arterial equilibration, $P_aCO_2 \approx P_ACO_2 \approx P_{ET}CO_2$, i.e.
  healthy lungs without appreciable ventilation–perfusion mismatch or shunt (Riley &
  Cournand, 1949; Slessarev et al., 2007).
- A linear hypercapnic ventilatory response over the working range ≈ 40–80 mmHg (Read, 1967;
  West, 2012).
- A single population-mean slope $S = 2.69$, which will not match any individual and varies
  with age, sex and body size (Patrick & Howard, 1972; Aitken et al., 1986).
- Published slopes are typically minute-ventilation slopes ($\Delta \dot V_E / \Delta P_aCO_2$);
  they are used here for the alveolar slope, valid while dead-space ventilation changes little.

The method is a deterministic first-order approximation, intended to harmonise reported
CO₂-challenge magnitudes across studies rather than to predict an individual subject's
arterial tension.

## References
*(all held in the Zotero library — CO2 Ventilation collection)*

1. Dalton, J. (1802). *Experimental Essays on the Constitution of Mixed Gases.* Memoirs of the
   Literary and Philosophical Society of Manchester, 5, 535–602.
2. Haldane, J. S., & Priestley, J. G. (1905). The regulation of the lung-ventilation.
   *The Journal of Physiology, 32*(3–4), 225–266. https://doi.org/10.1113/jphysiol.1905.sp001081
3. Riley, R. L., & Cournand, A. (1949). Ideal alveolar air and the analysis of
   ventilation-perfusion relationships in the lungs. *Journal of Applied Physiology, 1*(12),
   825–847. https://doi.org/10.1152/jappl.1949.1.12.825
4. Nielsen, M., & Smith, H. (1952). Studies on the regulation of respiration in acute hypoxia.
   *Acta Physiologica Scandinavica, 24*(4), 293–313.
   https://doi.org/10.1111/j.1748-1716.1952.tb00847.x
5. Read, D. J. C. (1967). A clinical method for assessing the ventilatory response to carbon
   dioxide. *Australasian Annals of Medicine, 16*(1), 20–32.
   https://doi.org/10.1111/imj.1967.16.1.20
6. Patrick, J. M., & Howard, A. (1972). The influence of age, sex, body size and lung size on
   the control and pattern of breathing during CO₂ inhalation in Caucasians.
   *Respiration Physiology, 16*(3), 337–350. https://doi.org/10.1016/0034-5687(72)90063-1
7. Hirshman, C. A., McCullough, R. E., & Weil, J. V. (1975). Normal values for hypoxic and
   hypercapnic ventilatory drives in man. *Journal of Applied Physiology, 38*(6), 1095–1098.
   https://doi.org/10.1152/jappl.1975.38.6.1095
8. Aitken, M. L., Franklin, J. L., Pierson, D. J., & Schoene, R. B. (1986). Influence of body
   size and gender on control of ventilation. *Journal of Applied Physiology, 60*(6),
   1894–1899. https://doi.org/10.1152/jappl.1986.60.6.1894
9. Mohan, R., & Duffin, J. (1997). The effect of hypoxia on the ventilatory response to carbon
   dioxide in man. *Respiration Physiology, 108*(2), 101–115.
   https://doi.org/10.1016/S0034-5687(97)00024-8
10. Pedersen, M. E. F., Fatemian, M., & Robbins, P. A. (1999). Identification of fast and slow
    ventilatory responses to carbon dioxide under hypoxic and hyperoxic conditions in humans.
    *The Journal of Physiology, 521*(1), 273–287.
    https://doi.org/10.1111/j.1469-7793.1999.00273.x
11. Cruickshank, S., & Hirschauer, N. (2004). The alveolar gas equation. *Continuing Education
    in Anaesthesia Critical Care & Pain, 4*(1), 24–27. https://doi.org/10.1093/bjaceaccp/mkh008
12. Slessarev, M., Han, J., Mardimae, A., Prisman, E., Preiss, D., Volgyesi, G., Ansel, C.,
    Duffin, J., & Fisher, J. A. (2007). Prospective targeting and control of end-tidal CO₂ and
    O₂ concentrations. *The Journal of Physiology, 581*(3), 1207–1219.
    https://doi.org/10.1113/jphysiol.2007.129395
13. West, J. B. (2012). *Respiratory Physiology: The Essentials* (9th ed.). Lippincott Williams
    & Wilkins.
14. Lumb, A. B., & Thomas, C. R. (2020). *Nunn's Applied Respiratory Physiology* (9th ed.).
    Elsevier.